# CSCI316: Big Data Mining - Modeling and Evaluation (Spark ML + PyTorch)
## Dubai Land Department Transaction Price Prediction

This notebook implements the **data preparation and modeling workflow**:
- Data loading and verification
- Domain-informed feature engineering
- Gradient Boosting ensemble (from scratch, Spark)
- Neural Network (MLP) training for tabular regression
- RMSE, MAE, and R² evaluation

**Input**: uses the prepared dataset `Transactions_copy.csv` (already cleaned/preprocessed).

## 1. Load Required Libraries

In [1]:
# --- Colab setup: run this cell first when using Google Colab ---
try:
  import google.colab
  _in_colab = True
except ImportError:
  _in_colab = False

if _in_colab:
  !pip install -q pyspark
  base_dir = "/content"  # Upload Transactions_copy.csv
  print("Colab detected: PySpark installed, base_dir = /content")
  print("Upload 'Transactions_copy.csv' to the file browser (left panel), or set base_dir to your Drive path.")
else:
  import os
  base_dir = os.getcwd()
  print("Local run: base_dir = current working directory")

Local run: base_dir = current working directory


In [2]:
import os
import math
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor

# Keep notebooks quieter
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Basic Data Verification

In [3]:
import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CSCI316-Modeling")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

checkpoint_dir = os.path.join(os.path.dirname(os.getcwd()), "checkpoints")
if not os.path.exists(checkpoint_dir): os.makedirs(checkpoint_dir)
spark.sparkContext.setCheckpointDir(checkpoint_dir)

k_folds = 10
base_dir = os.path.dirname(os.getcwd()) 
input_csv = os.path.join(base_dir, "data", "Transactions_copy.csv")

df = spark.read.option("header", "true").option("inferSchema", "true").csv(input_csv)
df = df.sample(False, 0.05, 42) 

print(f"Safe Session started. Loaded {df.count():,} rows.")

The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.

/Users/muhammadtariq/Downloads/CSCI316_PricePulseAI_Project1_Deliverable/.venv/lib/python3.10/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

## 3. Feature Engineering (Domain-Informed)

### 3.1 Filter to Residential Sales Only

In [ ]:

before = df.count()
df_residential = df.filter(F.col("is_residential") == 1)
after = df_residential.count()

print(f"Original dataset size: {before:,}")
print(f"After filtering to residential only: {after:,}")
print(f"Rows removed: {before - after:,} ({100 * (before - after) / max(before,1):.1f}%)")

usage_cols = [
    "is_commercial",
    "is_industrial",
    "is_hospitality",
    "is_storage",
    "is_agricultural",
    "is_other_usage",
]
usage_cols = [c for c in usage_cols if c in df_residential.columns]
if usage_cols:
    df_residential = df_residential.drop(*usage_cols)

print(f"\nColumns after dropping redundant usage flags: {len(df_residential.columns)}")
df = df_residential.cache()

Original dataset size: 94,435
After filtering to residential only: 80,791
Rows removed: 13,644 (14.4%)

Columns after dropping redundant usage flags: 17


### 3.2 Derive Price-Per-Sqm Target (More Stable than Raw Price)

In [ ]:

df = df.withColumn("actual_price", F.exp(F.col("actual_worth")))


df = df.filter(F.col("procedure_area") > 0)
df = df.withColumn("price_per_sqm", F.col("actual_price") / F.col("procedure_area"))

before = df.count()

q1, q99 = df.approxQuantile("price_per_sqm", [0.01, 0.99], 0.001)
df = df.filter((F.col("price_per_sqm") >= F.lit(q1)) & (F.col("price_per_sqm") <= F.lit(q99)))

after = df.count()
print(f"Rows before outlier removal: {before:,}")
print(f"Rows after outlier removal (1st-99th percentile): {after:,}")

summary = df.select(
    F.min("price_per_sqm").alias("min"),
    F.max("price_per_sqm").alias("max"),
    F.mean("price_per_sqm").alias("mean"),
    F.expr("percentile_approx(price_per_sqm, 0.5)").alias("median"),
    F.stddev("price_per_sqm").alias("std"),
).toPandas().iloc[0]

print("\nPrice-Per-Sqm Target Summary (AED/sqm):")
print(f"  Min: {summary['min']:,.2f}")
print(f"  Max: {summary['max']:,.2f}")
print(f"  Mean: {summary['mean']:,.2f}")
print(f"  Median: {summary['median']:,.2f}")
print(f"  Std: {summary['std']:,.2f}")

df = df.cache()

Rows before outlier removal: 80,791
Rows after outlier removal (1st-99th percentile): 79,127

Price-Per-Sqm Target Summary (AED/sqm):
  Min: 502.31
  Max: 44,366.64
  Mean: 13,002.79
  Median: 11,405.27
  Std: 7,783.27


In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

area_stats = df.groupBy("area_name", "sale_year", "sale_quarter").agg(F.mean("price_per_sqm").alias("area_mean_price"))
df = df.join(area_stats, on=["area_name", "sale_year", "sale_quarter"], how="left")

high_card = ["area_name", "master_project"]
global_m = df.agg(F.mean("price_per_sqm")).first()[0]
for col in high_card:
    df = df.join(df.groupBy(col).agg(F.mean("price_per_sqm").alias(f"{col}_encoded")), on=col, how="left").fillna(float(global_m))

low_card = ["property_type", "property_sub_type", "reg_type", "procedure_group"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in low_card]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in low_card]
df_cat = Pipeline(stages=indexers + encoders).fit(df).transform(df)

num_cols = ["procedure_area", "bedrooms", "has_parking", "is_penthouse", "area_mean_price", "area_name_encoded", "master_project_encoded"]
df_cat = df_cat.fillna(0, subset=num_cols) 

assembler = VectorAssembler(inputCols=num_cols + [f"{c}_ohe" for c in low_card], outputCol="features", handleInvalid="keep")
df_model = assembler.transform(df_cat).select("features", F.col("price_per_sqm").alias("label")).withColumn("row_id", F.monotonically_increasing_id()).cache()
df_model.count()
print("Features ready.")

Market-history features created.
Rows: 79,127

Market-History Feature Statistics:
+-------+------------------+------------------+----------------------+--------------------+
|summary|   area_mean_price| area_median_price|area_transaction_count|       area_momentum|
+-------+------------------+------------------+----------------------+--------------------+
|  count|             79127|             79127|                 79127|               79127|
|   mean|13002.788215101658|12723.403139248194|     68.88721928039733|0.022598234857184627|
| stddev| 6307.886114289927| 6465.571083056302|     67.37110754470825|  0.4308897682337968|
|    min| 502.3149510243627| 502.3149510243627|                     1| -0.9748993208970116|
|    max| 44228.21981424152| 44228.21981424152|                   347|  51.322701903160066|
+-------+------------------+------------------+----------------------+--------------------+


High-cardinality features (target-encode): ['area_name', 'master_project']
Low-cardinali

## 4. Modeling Setup

This section prepares cross-validation utilities and a scratch Gradient Boosting implementation.

In [ ]:
import numpy as np

# Keep a deterministic seed for reproducibility
np.random.seed(42)

def add_kfold_column_balanced(df_in, k=10, seed=42):
    return df_in.withColumn("fold", (F.abs(F.hash(F.col("row_id"), F.lit(seed))) % k).cast("int"))

def evaluate_regression(pred_df):
    pred_df = pred_df.withColumn("label_mean", F.avg("label").over(Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)))
    sse = F.sum(F.pow(F.col("label") - F.col("prediction"), 2))
    sst = F.sum(F.pow(F.col("label") - F.col("label_mean"), 2))

    m = pred_df.select(
        F.sqrt(F.avg(F.pow(F.col("prediction") - F.col("label"), 2))).alias("rmse"),
        F.avg(F.abs(F.col("prediction") - F.col("label"))).alias("mae"),
        F.when(sst > 0, F.lit(1.0) - (sse / sst)).otherwise(F.lit(0.0)).alias("r2")
    ).first()
    return float(m["rmse"]), float(m["mae"]), float(m["r2"])

def cv_manual_gradient_boosting(df_in, k=10, n_estimators=10, learning_rate=0.1, max_depth=5, sample_fraction=1.0, model_name="Gradient Boosting (Scratch, Spark)"):
    r_list, m_list, r2_list = [], [], []

    for f in range(k):
        tr = df_in.filter(F.col("fold") != f).select("row_id", "features", "label").checkpoint()
        te = df_in.filter(F.col("fold") == f).select("row_id", "features", "label").checkpoint()

        tr_pred = tr.select("row_id", "label").withColumn("prediction", F.lit(0.0))
        te_pred = te.select("row_id", "label").withColumn("prediction", F.lit(0.0))

        for i in range(n_estimators):
            residual_df = (
                tr.join(tr_pred.select("row_id", F.col("prediction").alias("pred_so_far")), on="row_id")
                  .withColumn("residual", F.col("label") - F.col("pred_so_far"))
                  .select("row_id", "features", F.col("residual").alias("label"))
            )

            if sample_fraction < 1.0:
                residual_df = residual_df.sample(False, sample_fraction, 42 + i)

            weak_learner = DecisionTreeRegressor(
                featuresCol="features",
                labelCol="label",
                maxDepth=max_depth,
                seed=42 + i,
            ).fit(residual_df)

            tr_update = weak_learner.transform(tr).select("row_id", (F.col("prediction") * F.lit(learning_rate)).alias("inc"))
            te_update = weak_learner.transform(te).select("row_id", (F.col("prediction") * F.lit(learning_rate)).alias("inc"))

            tr_pred = tr_pred.join(tr_update, on="row_id").withColumn("prediction", F.col("prediction") + F.col("inc")).drop("inc")
            te_pred = te_pred.join(te_update, on="row_id").withColumn("prediction", F.col("prediction") + F.col("inc")).drop("inc")

        rmse, mae, r2 = evaluate_regression(te_pred.select("label", "prediction"))
        r_list.append(rmse)
        m_list.append(mae)
        r2_list.append(r2)
        print(f"{model_name} - Fold {f+1}/{k}: RMSE={rmse:,.2f} AED/sqm, MAE={mae:,.2f} AED/sqm, R²={r2:.4f}")

        tr.unpersist()
        te.unpersist()

    print("\n" + "="*70)
    print(f"{model_name} Results ({k}-Fold Cross-Validation):")
    print("="*70)
    print(f"  Average RMSE: {np.mean(r_list):,.2f} (+/- {np.std(r_list):,.2f}) AED/sqm")
    print(f"  Average MAE:  {np.mean(m_list):,.2f} (+/- {np.std(m_list):,.2f}) AED/sqm")
    print(f"  Average R²:   {np.mean(r2_list):.4f} (+/- {np.std(r2_list):.4f})")
    print("="*70 + "\n")

    return {
        "avg_rmse": float(np.mean(r_list)),
        "std_rmse": float(np.std(r_list)),
        "avg_mae": float(np.mean(m_list)),
        "std_mae": float(np.std(m_list)),
        "avg_r2": float(np.mean(r2_list)),
        "std_r2": float(np.std(r2_list)),
        "model_name": model_name,
    }

k_folds = 10
cv_df = add_kfold_column_balanced(df_model, k=k_folds).cache()
cv_df.count()
print("Modeling utilities and CV dataset ready.")

All Spark CV + ensemble helpers defined successfully.


## 5. Model Training and Evaluation

### 5.1 Gradient Boosting Ensemble

> Memory note: in constrained environments, lighter settings are used automatically.

In [ ]:
print("\n" + "="*60)
print("TRAINING GRADIENT BOOSTING ENSEMBLE (from scratch, residual boosting)")
print("="*60 + "\n")

_lighter_gb = globals().get("_in_colab", False)
if _lighter_gb:
    print("Using memory-friendly GB settings (Colab): 5 trees, depth 4, 40% subsample\n")

gb_results = cv_manual_gradient_boosting(
    cv_df,
    k=k_folds,
    n_estimators=5 if _lighter_gb else 10,
    learning_rate=0.1,
    max_depth=4 if _lighter_gb else 5,
    sample_fraction=0.4 if _lighter_gb else 1.0,
    model_name="Gradient Boosting (Scratch, Spark)",
)

## 6. Neural Network Model (MLP for Tabular Data)

### Architecture:
- Input layer with number of features
- Dense(128) + ReLU + Dropout(0.3)
- Dense(64) + ReLU
- Dense(32) + ReLU
- Dense(1) output for regression

### Training:
- PyTorch with GPU support (falls back to CPU)
- DataLoader for mini-batch training
- MSELoss with Adam optimizer
- Early stopping on validation loss

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Convert Spark DataFrame to Pandas and prepare data
print("\nPreparing data for Neural Network...")
df_pandas = df_model.select("features", "label").toPandas()

# Extract features and labels
X = df_pandas["features"].apply(lambda x: x.toArray()).apply(pd.Series).values
y = df_pandas["label"].values.reshape(-1, 1)

# Split data into train and validation sets (80-20)
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Scale target variable for neural network stability
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_val_scaled = y_scaler.transform(y_val)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
y_train_tensor = torch.FloatTensor(y_train_scaled).to(device)
X_val_tensor = torch.FloatTensor(X_val_scaled).to(device)
y_val_tensor = torch.FloatTensor(y_val_scaled).to(device)

# Create DataLoaders
batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")


In [ ]:
# Define MLP Architecture
class MLPRegressor(nn.Module):
    def __init__(self, input_size, hidden1=128, hidden2=64, hidden3=32, dropout=0.3):
        super(MLPRegressor, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        
        self.fc3 = nn.Linear(hidden2, hidden3)
        self.relu3 = nn.ReLU()
        
        self.fc_out = nn.Linear(hidden3, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.relu2(x)
        
        x = self.fc3(x)
        x = self.relu3(x)
        
        x = self.fc_out(x)
        return x

# Initialize model
input_size = X_train.shape[1]
model = MLPRegressor(input_size=input_size, hidden1=128, hidden2=64, hidden3=32, dropout=0.3).to(device)
print(f"\nModel Architecture:")
print(model)


In [ ]:
# Training loop with early stopping
def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, patience=10):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    print(f"\n{'='*60}")
    print(f"TRAINING NEURAL NETWORK MLP")
    print(f"{'='*60}\n")
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}")
                model.load_state_dict(best_model_state)
                break
    
    print(f"\nTraining completed. Final Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}\n")
    return model, train_losses, val_losses

# Train the model
print("Training Neural Network...")
model, train_losses, val_losses = train_model(
    model, 
    train_loader, 
    val_loader, 
    epochs=50,
    lr=0.001,
    patience=10
)


In [ ]:
# Evaluate Neural Network
def evaluate_nn_model(model, X_val, y_val, y_val_scaled, device, y_scaler):
    model.eval()
    X_tensor = torch.FloatTensor(X_val).to(device)
    
    with torch.no_grad():
        predictions_scaled = model(X_tensor).cpu().numpy()
    
    # Inverse transform predictions and actual values back to original scale
    predictions = y_scaler.inverse_transform(predictions_scaled)
    
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    mae = mean_absolute_error(y_val, predictions)
    r2 = r2_score(y_val, predictions)
    
    return rmse, mae, r2, predictions

# Evaluate on validation set
print(f"{'='*60}")
print(f"NEURAL NETWORK EVALUATION")
print(f"{'='*60}\n")

nn_rmse, nn_mae, nn_r2, nn_predictions = evaluate_nn_model(model, X_val_scaled, y_val, y_val_scaled, device, y_scaler)

print(f"Validation Set Metrics:")
print(f"  RMSE: {nn_rmse:,.2f} AED/sqm")
print(f"  MAE: {nn_mae:,.2f} AED/sqm")
print(f"  R² Score: {nn_r2:.4f}\n")

# Store results
nn_results = {
    "model_name": "Neural Network (MLP)",
    "rmse": nn_rmse,
    "mae": nn_mae,
    "r2": nn_r2
}


## 7. Model Comparison (Gradient Boosting vs Neural Network)

In [ ]:
print(f"{'='*90}")
print("COMPARISON: GRADIENT BOOSTING vs NEURAL NETWORK")
print(f"{'='*90}\n")

comparison_data = [
    {
        "Model": "Gradient Boosting",
        "RMSE (AED/sqm)": f"{gb_results['avg_rmse']:,.2f}",
        "RMSE Std": f"{gb_results['std_rmse']:,.2f}",
        "MAE (AED/sqm)": f"{gb_results['avg_mae']:,.2f}",
        "MAE Std": f"{gb_results['std_mae']:,.2f}",
        "R² Score": f"{gb_results['avg_r2']:.4f}",
        "R² Std": f"{gb_results['std_r2']:.4f}",
        "Approach": "Spark ML",
    },
    {
        "Model": "Neural Network (MLP)",
        "RMSE (AED/sqm)": f"{nn_rmse:,.2f}",
        "RMSE Std": "Train/Val Split",
        "MAE (AED/sqm)": f"{nn_mae:,.2f}",
        "MAE Std": "Train/Val Split",
        "R² Score": f"{nn_r2:.4f}",
        "R² Std": "Train/Val Split",
        "Approach": "PyTorch",
    },
]

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print(f"\n{'='*90}\n")

print("KEY INSIGHTS:")
print(f"  • Gradient Boosting RMSE: {gb_results['avg_rmse']:,.2f} AED/sqm")
print(f"  • Gradient Boosting R²: {gb_results['avg_r2']:.4f}")
print(f"  • Neural Network RMSE: {nn_rmse:,.2f} AED/sqm")
print(f"  • Neural Network R² Score: {nn_r2:.4f}")
print("\nNote: Spark model uses 10-fold CV; NN uses train/val split (80/20)\n")

## 8. Summary

In [ ]:
print("\n" + "=" * 100)
print("FINAL RESULTS SUMMARY")
print("=" * 100)

print("\nGradient Boosting (Spark ML - 10-Fold CV):")
print(f"  Avg RMSE: {gb_results['avg_rmse']:,.2f} AED/sqm")
print(f"  Std RMSE: {gb_results['std_rmse']:,.2f} AED/sqm")
print(f"  Avg MAE: {gb_results['avg_mae']:,.2f} AED/sqm")
print(f"  Std MAE: {gb_results['std_mae']:,.2f} AED/sqm")
print(f"  Avg R²: {gb_results['avg_r2']:.4f}")
print(f"  Std R²: {gb_results['std_r2']:.4f}")

print("\nNeural Network MLP (PyTorch - Train/Val Split):")
print(f"  RMSE: {nn_rmse:,.2f} AED/sqm")
print(f"  MAE: {nn_mae:,.2f} AED/sqm")
print(f"  R² Score: {nn_r2:.4f}")

print(f"\n{'='*100}")


FINAL RESULTS SUMMARY (10-Fold Cross-Validation, Spark)
                             Model  Avg RMSE (AED/sqm)  Std RMSE  Avg MAE (AED/sqm)   Std MAE
  Baseline (Single Tree, Spark ML)         3710.281767 96.927338        2363.598861 44.212377
 Bagging Ensemble (Scratch, Spark)         3273.043120 69.234113        2139.709086 30.780701
Gradient Boosting (Scratch, Spark)         5732.314633 66.809899        4395.891204 45.503133


IMPROVEMENT OVER BASELINE (RMSE):
  Bagging Ensemble: +11.78%
  Gradient Boosting: -54.50%

PREDICTION CONSISTENCY (Lower Std = More Stable):
  Baseline Std RMSE: 96.93 AED/sqm
  Bagging Std RMSE: 69.23 AED/sqm
  Gradient Boosting Std RMSE: 66.81 AED/sqm


## 9. Notes

### What this notebook includes:
1. Data loading and preprocessing pipeline
2. Feature engineering for tabular inputs
3. Gradient Boosting (scratch, Spark)
4. Neural Network (MLP) training and evaluation

### Evaluation Metrics:
- RMSE (AED/sqm)
- MAE (AED/sqm)
- R² score (for MLP)

### Runtime Guidance:
- Run all cells from top to bottom after restarting the kernel.
- If the kernel is restarted, rerun preprocessing cells before model training.